# 03. Causal inference

この notebook は因果推論 stage を扱います。

主題:

- edge-weight mode と treatment-effect mode の違い
- treatment / outcome / estimand / adjustment set
- ATE と ATT
- g-computation、IPW、AIPW、OLS、差分
- overlap、bad controls、feature semantics validation

最重要点: discovery graph の edge weight は ATE/ATT ではありません。ATE/ATT が欲しい場合は treatment-effect mode で estimand を明示します。

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPOSITORY_MARKER = "pyproject.toml"
cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / REPOSITORY_MARKER).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("repository root was not found")

ARTICLE_ROOT = PROJECT_ROOT
SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
for path in (SRC_DIR,):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

ARTICLE_ROOT

## 2 つの inference mode

| mode | 入力 | 出力 | 解釈 |
|---|---|---|---|
| `edge_weight` | discovery graph の `edges.csv` と inference frame | 各 edge の局所線形係数 | graph edge の重み。ATE/ATT ではない |
| `treatment_effect` | treatment、outcome、estimand、adjustment set | ATE/ATT の推定値 | 識別仮定に条件づく treatment effect |

edge-weight は探索結果の後処理です。treatment-effect は明示的な因果問いへの推定です。

In [ ]:
from causal_atelier.infrastructure.config import load_yaml_mapping
from causal_atelier.preprocessing.common import FeatureSemanticsCatalog

inference_config_path = PROJECT_ROOT / "configs" / "causal" / "inference" / "defaults.yaml"
semantics_path = PROJECT_ROOT / "configs" / "preprocessing" / "feature_semantics.yaml"
design_path = PROJECT_ROOT / "configs" / "causal" / "inference" / "designs" / "completejourney_household.yaml"

inference_config = load_yaml_mapping(inference_config_path)
causal_design = load_yaml_mapping(design_path)
semantics = FeatureSemanticsCatalog.from_mapping(load_yaml_mapping(semantics_path))

pd.DataFrame([{
    "default_mode": inference_config["mode"],
    "treatment": inference_config["treatment_effect"]["treatment"],
    "outcome": inference_config["treatment_effect"]["outcome"],
    "estimand": inference_config["treatment_effect"]["estimand"],
    "adjustment_strategy": inference_config["treatment_effect"]["adjustment_strategy"],
}])

## Feature semantics と bad controls

調整変数は何でも入れればよいわけではありません。

- treatment を調整に入れるのは不正。
- outcome を調整に入れるのは不正。
- mediator を total effect 推定で調整すると estimand が変わる。
- collider を調整すると collider bias が開く可能性がある。
- post-treatment variable は交絡調整には原則入れない。

この pipeline は `FeatureSemanticSpec` を使って、adjustment set に明らかな bad controls が混ざることを error にします。

In [ ]:
semantics_table = pd.DataFrame([feature.to_dict() for feature in semantics.features])
semantics_table.loc[:, ["name", "role", "allowed_for_adjustment", "post_treatment"]].head(25)

## 合成データで ATE を推定する

真の効果が分かっているデータ生成過程を作ります。

- `x` は treatment と outcome の両方に影響する交絡変数。
- `treated` は `x` に依存して割り当てられる。
- `outcome = 1 + 2 * treated + 0.7 * x + noise`。

真の ATE は 2.0 です。未調整差分は、交絡のため 2.0 からずれます。

In [ ]:
from causal_atelier.causal.inference.estimators.treatment_effect import TreatmentEffectEstimator

rng = np.random.default_rng(123)
n = 5000
x = rng.normal(size=n)
propensity = 1.0 / (1.0 + np.exp(-0.5 * x))
treated = rng.binomial(1, propensity)
outcome = 1.0 + 2.0 * treated + 0.7 * x + rng.normal(scale=0.2, size=n)
synthetic = pd.DataFrame({"treated": treated, "outcome": outcome, "x": x})

estimator = TreatmentEffectEstimator(
    synthetic,
    treatment="treated",
    outcome="outcome",
    covariates=["x"],
    estimand="ATE",
    propensity_clip=(0.01, 0.99),
)
ate = estimator.estimate([
    "diff_in_means",
    "ols_coefficient",
    "g_computation_ate",
    "ipw_ate",
    "aipw_ate",
])
ate.loc[:, ["method", "estimand", "effect", "std_error", "ci_low", "ci_high", "notes"]]

## ATT も別 estimand

ATE は全体平均効果、ATT は treated population における平均効果です。効果が heterogeneous な場合、ATE と ATT は一致しません。

この合成データでは効果を constant 2.0 にしているため、ATE と ATT は近い値になります。

In [ ]:
att_estimator = TreatmentEffectEstimator(
    synthetic,
    treatment="treated",
    outcome="outcome",
    covariates=["x"],
    estimand="ATT",
    propensity_clip=(0.01, 0.99),
)
att = att_estimator.estimate(["g_computation_att", "ipw_att", "aipw_att"])
att.loc[:, ["method", "estimand", "effect", "std_error", "ci_low", "ci_high", "notes"]]

## Overlap warning

positivity / overlap は、各 covariate profile で treatment と control の両方が観測されるという要求です。propensity が 0 または 1 に近い領域では、IPW や AIPW は不安定になります。

In [ ]:
x_extreme = rng.normal(size=1000)
extreme = pd.DataFrame({
    "treated": (x_extreme > 0).astype(int),
    "outcome": 1.0 + 2.0 * (x_extreme > 0).astype(int) + x_extreme + rng.normal(scale=0.2, size=1000),
    "x": x_extreme,
})
extreme_estimator = TreatmentEffectEstimator(
    extreme,
    treatment="treated",
    outcome="outcome",
    covariates=["x"],
    estimand="ATE",
    propensity_clip=(0.05, 0.95),
)
extreme_estimator.estimate(["ipw_ate"]).loc[:, ["method", "effect", "notes"]]

## 実務上の読み方

- まず causal design で treatment、outcome、estimand、time zero を明示する。
- feature semantics で treatment/outcome/covariate/post-treatment を確認する。
- adjustment set は causal design に基づいて決める。機械的な変数選択だけに任せない。
- overlap、balance、sample size、model dependence を診断する。
- 推定結果は識別仮定に条件づく。コードが仮定を証明するわけではない。

代替仮説: discovery graph を使って adjustment set を自動で選べば十分、という考え方もあります。しかし graph が誤っている場合、mediator や collider を入れる危険があります。そのため、この pipeline では semantics validation と causal design を別に置いています。